# DePaso · IA — Paso 2: Entrenamiento

**Qué hace:** entrena el clasificador MobileNetV2 (transfer learning en 2 etapas) con el dataset del Paso 1. **Dual input**: imagen 224×224 + flag `has_reference_object`.

**Dónde corre:** Colab con **GPU** (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`). ~30-60 min con el dataset ~2000 del Paso 1.

**Salida (Drive `MyDrive/depaso_ml/models/`):** `cargo_classifier_v1.keras`, `test_split.csv`, `metadata.json` (con `version: v2`).

## 0. Verificar GPU

Si esta celda no muestra una GPU, andá a *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU* y volvé a correr.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs disponibles:', gpus)
assert gpus, 'No hay GPU. Cambiá el tipo de entorno a T4 GPU y reiniciá.'
!nvidia-smi -L

## 1. Setup

In [ ]:
# Montar Google Drive (acá viven el dataset y el modelo, para que no se pierdan
# cuando Colab reinicia el runtime).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clonar el repo (trae los scripts del pipeline) y fijar las rutas de trabajo.
import os
REPO_URL = 'https://github.com/martinatoffoletto/DePaso.git'
if not os.path.exists('/content/DePaso'):
    !git clone {REPO_URL} /content/DePaso
else:
    !cd /content/DePaso && git pull

ML_DIR    = '/content/DePaso/depaso_rest/ml'          # scripts del pipeline
# El dataset y el modelo viven en Drive para sobrevivir reinicios del runtime:
WORK      = '/content/drive/MyDrive/depaso_ml'
RAW_DIR   = f'{WORK}/dataset/raw'
CLEAN_DIR = f'{WORK}/dataset/clean'
MODEL_DIR = f'{WORK}/models'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('ML_DIR   :', ML_DIR)
print('RAW_DIR  :', RAW_DIR)
print('CLEAN_DIR:', CLEAN_DIR)
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
# Colab ya trae TensorFlow; esto solo asegura una versión compatible.
!pip install -q 'tensorflow>=2.17.0' pandas scikit-learn

## 2. Entrenar

- **Etapa A**: base MobileNetV2 congelada, entrena la cabeza (hasta 40 épocas, lr=1e-3).

- **Etapa B**: descongela las últimas 60 capas, fine-tuning (hasta 25 épocas, lr=1e-5).

- EarlyStopping (patience=8, `restore_best_weights`) + ReduceLROnPlateau sobre `val_loss`: el número de épocas es un techo, cortan/bajan el lr solos.

- `class_weight` balanceado del train + label smoothing 0.1 (despegan a `m` del rol de aspirador del v1). Augmentation sobre píxeles crudos en el pipeline de datos.

El script respeta los constraints intocables: `CATEGORIES = ['s','m','l','xl']` en orden fijo, dos inputs siempre, y `preprocess_input` solo en el pipeline de datos.

> Si imprime `⚠️ solo N imágenes con has_reference_object=1`, el 2º input está casi muerto: sumá fotos con objeto de referencia (Paso 1, §4) y re-corré el Paso 1.

In [ ]:
!python {ML_DIR}/train_classifier.py --data-dir {CLEAN_DIR} --out-dir {MODEL_DIR}

## 3. Revisar el resultado

Objetivo de accuracy: **≥80%** (mínimo defendible en la tesis: **75%**). Si quedó por debajo, sumá más fotos propias de las clases/condiciones flojas y reentrená.

In [ ]:
import json
meta = json.load(open(f'{MODEL_DIR}/metadata.json'))
print(json.dumps(meta, indent=2, ensure_ascii=False))
acc = meta['best_val_accuracy']
print()
if acc >= 0.80:  print(f'✅ val_accuracy = {acc:.2%} — objetivo alcanzado.')
elif acc >= 0.75: print(f'🟡 val_accuracy = {acc:.2%} — defendible, pero sumá datos si podés.')
else:            print(f'🔴 val_accuracy = {acc:.2%} — por debajo del mínimo. Más fotos y reentrenar.')

## ✅ Listo

El modelo quedó en `MyDrive/depaso_ml/models/`. Seguí con **`03_evaluacion.ipynb`** para el análisis de sesgos (las figuras van directo a la tesis).